In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

pd.set_option("display.max_columns",50)
pd.set_option("display.width",200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

#Shared colour pallete so every chart reads consistently
PALETTE =   px.colors.qualitative.Bold
PRIMARY = "#1f77b4"
ACCENT = "#FF7F0E"

In [ ]:
cleaned_customer_transaction = pd.read_csv("../customer_data/artifacts/cleaned_transaction_data.csv")

In [ ]:
cleaned_customer_transaction.head(10)

In [ ]:
cleaned_customer_transaction.info()

In [ ]:
cleaned_customer_transaction.isna().sum()

In [ ]:
#Parse  the data once, reuse everywhere 
cleaned_customer_transaction["TransactionDate"] = pd.to_datetime(cleaned_customer_transaction["TransactionDate"])
cleaned_customer_transaction["CustomerDOB"]= pd.to_datetime(cleaned_customer_transaction["CustomerDOB"])
cleaned_customer_transaction["TransactionTime"] = pd.to_datetime(cleaned_customer_transaction["TransactionTime"], format = "%H:%M:%S").dt.time

print(f"Rows: {len(cleaned_customer_transaction):,}")
print(f"Columns: {cleaned_customer_transaction.shape[1]}")
print(f"Customers: {cleaned_customer_transaction["CustomerID"].nunique():,}")
print(f"Date range: {cleaned_customer_transaction["TransactionDate"].min().date()} - {cleaned_customer_transaction["TransactionDate"].max().date()}")
print(f"Total nulls: {cleaned_customer_transaction.isna().sum().sum()}")

In [ ]:
cleaned_customer_transaction["TransactionStatus"].value_counts()

In [ ]:
# GBP equivalent amount (needed formonetary charts;cryptorows are in asset units)
FX_TO_GBP = {"GBP": 1.0, "EUR": 0.85, "USD": 0.78, "USDT": 0.78,
             "BTC": 42_000.0, "ETH": 2_300.0}

cleaned_customer_transaction["AmountGBP"]= cleaned_customer_transaction["TransactionAmount"] * cleaned_customer_transaction["TransactionCurrency"].map(FX_TO_GBP)

cleaned_customer_transaction["RevenueGBP"] = np.where(
    cleaned_customer_transaction["TransactionStatus"] == "Completed",
    cleaned_customer_transaction["AmountGBP"],
    0.0,
)

SNAPSHOT = cleaned_customer_transaction["TransactionDate"].max()

cleaned_customer_transaction["Year"] = cleaned_customer_transaction["TransactionDate"].dt.year
cleaned_customer_transaction["Month"] = cleaned_customer_transaction["TransactionDate"].dt.to_period("M").astype(str)
cleaned_customer_transaction["DayOfWeek"] = cleaned_customer_transaction["TransactionDate"].dt.day_name()
cleaned_customer_transaction["Hour"] = cleaned_customer_transaction["TransactionTime"].apply(lambda t: t.hour)

cleaned_customer_transaction["Age"] = ((SNAPSHOT - cleaned_customer_transaction["CustomerDOB"]).dt.days / 365.25).round(1)
print("Derived column added")

In [ ]:
fig= px.histogram(
    cleaned_customer_transaction, x="AmountGBP", nbins = 80,
    title= "Distribution of Transaction Amount (GBP equivalent)",
    labels = {"AmountGBP": "Amount (GBP)"},
    color_discrete_sequence =[PRIMARY],
)
fig.update_layout(bargap=0.02, height=450)
fig.update_yaxes(title= "Number of transactions", type= "log")
fig.show()

In [ ]:
cleaned_customer_transaction["log_Amount"] = np.log10(cleaned_customer_transaction["AmountGBP"].clip(lower=0.01))

fig = px.histogram(
    cleaned_customer_transaction, x="log_Amount", nbins=80,
    title = "Distriution of log10(Transaction Amount) - GBP",
    labels={"log_Amount": "log10(Amount in GBP)"},
    color_discrete_sequence= [PRIMARY],
)
fig.update_layout(bargap=0.02, height=450)
fig.show()

In [ ]:
unique_cust = cleaned_customer_transaction.drop_duplicates("CustomerID")[["CustomerID", "Age", "CustGender", "CustLocation"]]
unique_cust.shape

fig = px.histogram(
    unique_cust, x="Age", nbins=40,
    title="Customer Age Distribution",
    color_discrete_sequence=[PRIMARY],
)

fig.update_layout(bargap=0.03, height=420)
fig.update_yaxes(title="Number of customers")
fig.show()

In [ ]:
gender_counts = unique_cust["CustGender"].value_counts().reset_index()
gender_counts.columns = ["Gender", "Count"]

fig = px.pie(
    gender_counts, names="Gender", values="Count",
    title="Customer Gender Split",
    color_discrete_sequence=PALETTE,
    hole=0.5,
)

fig.update_traces(textinfo="label+percent")
fig.show()


In [ ]:
import plotly.express as px

agg = (
    cleaned_customer_transaction
    .groupby("TransactionType", as_index=False)
    .agg(TotalAmount=("RevenueGBP", "sum"),
         Count=("RevenueGBP", "size"))
    .sort_values("TotalAmount", ascending=False)
)

fig = px.bar(
    agg, x="TransactionType", y="TotalAmount",
    title="Total Transaction Value by Type (GBP equivalent)",
    text="TotalAmount",
    color="TransactionType",
    color_discrete_sequence=PALETTE,
    hover_data={"Count": ":,"},
)

fig.update_traces(texttemplate="£%{text:,.0f}", textposition="outside")
fig.update_layout(height=500, showlegend=False)
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(title="Total value (GBP)")
fig.show()

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Customer age distribution",
        "Gender split",
        "Top 15 customer locations",
        "Wallet / crypto share of transactions",
    ),
    specs=[[{"type": "xy"}, {"type": "domain"}],
           [{"type": "xy"}, {"type": "domain"}]],
    vertical_spacing=0.20, horizontal_spacing=0.15,
)

# 1. Age histogram — go.Histogram, NOT px.histogram
fig.add_trace(
    go.Histogram(x=unique_cust["Age"], nbinsx=40, marker_color=PRIMARY),
    row=1, col=1,
)

# 2. Gender pie
gender = unique_cust["CustGender"].value_counts()
fig.add_trace(
    go.Pie(labels=gender.index, values=gender.values, hole=0.5,
           marker=dict(colors=PALETTE), textinfo="label+percent"),
    row=1, col=2,
)

# 3. Top 15 locations
loc = unique_cust["CustLocation"].value_counts().head(15).reset_index()
loc.columns = ["Location", "Customers"]
fig.add_trace(
    go.Bar(x=loc["Customers"], y=loc["Location"], orientation="h",
           marker_color=PRIMARY),
    row=2, col=1,
)

# 4. Wallet activity pie
wallet = cleaned_customer_transaction["WalletActivity"].value_counts()
labels = wallet.index.map({0: "Traditional", 1: "Wallet / Crypto"})
fig.add_trace(
    go.Pie(labels=labels, values=wallet.values, hole=0.5,
           marker=dict(colors=[PALETTE[0], PALETTE[1]]),
           textinfo="label+percent"),
    row=2, col=2,
)

fig.update_yaxes(autorange="reversed", row=2, col=1)
fig.update_xaxes(title="Age", row=1, col=1)
fig.update_layout(height=750, title_text="Customer and Channel Composition",
                  showlegend=False)
fig.show()


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Transaction status distribution",
        "Transaction type frequency",
        "Currency distribution",
        "Asset type distribution",
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12,
)

status = cleaned_customer_transaction["TransactionStatus"].value_counts()
fig.add_trace(
    go.Bar(x=status.index, y=status.values, marker_color=PALETTE[:len(status)],
           text=status.values, texttemplate="%{text:,}", textposition="outside"),
    row=1, col=1,
)

ttype = cleaned_customer_transaction["TransactionType"].value_counts()
fig.add_trace(
    go.Bar(x=ttype.index, y=ttype.values, marker_color=PRIMARY,
           text=ttype.values, texttemplate="%{text:,}", textposition="outside"),
    row=1, col=2,
)

cur = cleaned_customer_transaction["TransactionCurrency"].value_counts()
fig.add_trace(
    go.Bar(x=cur.index, y=cur.values, marker_color=ACCENT,
           text=cur.values, texttemplate="%{text:,}", textposition="outside"),
    row=2, col=1,
)

asset = cleaned_customer_transaction["AssetType"].value_counts()
fig.add_trace(
    go.Bar(x=asset.index, y=asset.values, marker_color=PALETTE[2],
           text=asset.values, texttemplate="%{text:,}", textposition="outside"),
    row=2, col=2,
)

fig.update_xaxes(tickangle=-30, row=1, col=2)
fig.update_xaxes(tickangle=-30, row=2, col=2)
fig.update_layout(height=750, title_text="Ledger Health Overview",
                  showlegend=False, bargap=0.25)
fig.show()

In [ ]:
monthly = cleaned_customer_transaction.groupby("Month").size().reset_index(name="Count")
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = (cleaned_customer_transaction.groupby("DayOfWeek").size()
       .reindex(dow_order).reset_index(name="Count"))
hourly = cleaned_customer_transaction.groupby("Hour").size().reset_index(name="Count")
heat = pd.crosstab(cleaned_customer_transaction["Hour"],
                   cleaned_customer_transaction["DayOfWeek"]).reindex(columns=dow_order)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Monthly transaction volume",
        "Day-of-week pattern",
        "Hour-of-day pattern",
        "Transaction density - hour × day",
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12,
)

fig.add_trace(
    go.Scatter(x=monthly["Month"], y=monthly["Count"], mode="lines+markers",
               line=dict(color=PRIMARY, width=3), fill="tozeroy",
               fillcolor="rgba(31,119,180,0.15)"),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(x=dow["DayOfWeek"], y=dow["Count"], marker_color=PRIMARY),
    row=1, col=2,
)

fig.add_trace(
    go.Scatter(x=hourly["Hour"], y=hourly["Count"], mode="lines+markers",
               line=dict(color=ACCENT, width=3)),
    row=2, col=1,
)

fig.add_trace(
    go.Heatmap(z=heat.values, x=heat.columns, y=heat.index,
               colorscale="Viridis", showscale=False),
    row=2, col=2,
)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(tickangle=-30, row=1, col=2)
fig.update_xaxes(dtick=2, title="Hour", row=2, col=1)
fig.update_yaxes(title="Hour", row=2, col=2)
fig.update_layout(height=750, title_text="Temporal Behaviour", showlegend=False)
fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# Cross-tab: rows = TransactionType, columns = TransactionStatus
status_by_type = pd.crosstab(
    cleaned_customer_transaction["TransactionType"],
    cleaned_customer_transaction["TransactionStatus"],
)


# Optional: normalise to percentages so success/failure mix is comparable across
# types with very different volumes. Comment out if you want raw counts.
status_by_type_pct = (
    status_by_type.div(status_by_type.sum(axis=1), axis=0) * 100
)


# Plotly grouped bar — one trace per status, one x-tick per transaction type
fig = go.Figure()

for i, status in enumerate(status_by_type.columns):
    fig.add_trace(
        go.Bar(
            name=status,
            x=status_by_type.index,
            y=status_by_type[status],
            marker_color=px.colors.qualitative.Bold[i % 8],
            hovertemplate=(
                "<b>%{x}</b><br>"           
                 f"Status: {status}<br>"
            "Count: %{y:,}<extra></extra>"
        ),
    )
)

fig.update_layout(
    barmode="group",
    title="Transaction Status by Transaction Type",
    xaxis_title="Transaction Type",
    yaxis_title="Number of Transactions",
    height=550,
    legend_title="Status",
)

fig.update_xaxes(tickangle=-30)
fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Create a cross-tabulation of currency and asset type
currency_asset = pd.crosstab(
    cleaned_customer_transaction["TransactionCurrency"],
    cleaned_customer_transaction["AssetType"]
)

# Create the figure
fig = go.Figure()

# Add one bar for each asset type
for i, asset in enumerate(currency_asset.columns):
    fig.add_trace(
        go.Bar(
            name=asset,
            x=currency_asset.index,
            y=currency_asset[asset],
            marker_color=px.colors.qualitative.Bold[i % 8]
        )
    )

# Update chart layout
fig.update_layout(
    title="Transaction Count by Currency and Asset Type",
    xaxis_title="Transaction Currency",
    yaxis_title="Number of Transactions",
    barmode="group",
    height=550,
    legend_title="Asset Type"
)

fig.show()

## 1. Transaction Status Distribution

### Why did we create this visualisation?
We created this visualisation to understand the outcomes of transactions processed through the Vault Chain Flow platform.

### What is it used to check?
It shows the number of transactions in each status category, including Completed, Pending, Failed, Rejected, and Under Review.

### Why is it important?
Understanding transaction outcomes helps identify operational issues and potential friction in the customer experience. For example, a high number of failed or rejected transactions may indicate problems that require further investigation. Pending and under-review transactions can also help identify processing delays.

This analysis provides an overview of transaction outcomes, although the reasons behind individual failures would require additional investigation.

In [ ]:
status_counts = (
    cleaned_customer_transaction["TransactionStatus"]
    .value_counts()
    .reset_index()
)

status_counts.columns = ["TransactionStatus", "Count"]

fig = px.bar(
    status_counts,
    x="TransactionStatus",
    y="Count",
    color="TransactionStatus",
    text="Count",
    title="Transaction Status Distribution",
    color_discrete_sequence=PALETTE
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    showlegend=False,
    height=500,
    xaxis_title="Transaction Status",
    yaxis_title="Number of Transactions"
)

fig.show()

## 2. Transaction Completion Rate by Transaction Type

### Why did we create this visualisation?
We created this visualisation to compare the proportion of completed transactions across different transaction types.

### What is it used to check?
It calculates the percentage of transactions marked Completed for each type, such as P2P Transfer, Wallet Transfer, Crypto Buy, Crypto Sell, Deposit, and Withdrawal.

### Why is it important?
Different transaction types may have different processing requirements and failure patterns. Comparing completion rates can help identify transaction types that may need further investigation or operational improvement.

Using percentages rather than raw counts allows us to compare transaction types more fairly, even when they have different numbers of transactions. A low completion rate does not, by itself, explain why transactions did not complete.

In [ ]:
transaction_success = (
    cleaned_customer_transaction
    .assign(
        IsCompleted=(
            cleaned_customer_transaction["TransactionStatus"]
            == "Completed"
        )
    )
    .groupby("TransactionType", as_index=False)
    .agg(
        TotalTransactions=("IsCompleted", "size"),
        CompletedTransactions=("IsCompleted", "sum")
    )
)

transaction_success["CompletionRate"] = (
    transaction_success["CompletedTransactions"]
    / transaction_success["TotalTransactions"]
    * 100
)

transaction_success = transaction_success.sort_values(
    "CompletionRate", ascending=True
)

fig = px.bar(
    transaction_success,
    x="CompletionRate",
    y="TransactionType",
    orientation="h",
    text="CompletionRate",
    title="Transaction Completion Rate by Type",
    color="CompletionRate",
    color_continuous_scale="Viridis",
    hover_data=[
        "TotalTransactions",
        "CompletedTransactions"
    ]
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    height=550,
    xaxis_title="Completion Rate (%)",
    yaxis_title="Transaction Type",
    xaxis_range=[0, 110],
    coloraxis_showscale=False
)

fig.show()

## Transaction Value Distribution by Asset Type

- **Purpose** — Compare the distribution of transaction values across different asset types.

- **AmountGBP** — The estimated GBP-equivalent value of each transaction. Enables:
  - Comparison of transaction value distributions across asset types
  - Identification of unusually large transaction values
  - Investigation of differences between traditional and crypto-related transaction values

- **Why it matters** — Helps the business understand transaction-value patterns and identify potential outliers for further investigation. Since crypto and traditional assets have different units and price characteristics, converting to a common currency provides a more meaningful comparison. The FX conversion assumptions should be reviewed before interpreting the results.

In [ ]:
fig = px.box(
    cleaned_customer_transaction,
    x="AssetType",
    y="AmountGBP",
    color="AssetType",
    title="Transaction Value Distribution by Asset Type",
    points=False,
    color_discrete_sequence=PALETTE
)

fig.update_layout(
    height=550,
    showlegend=False,
    xaxis_title="Asset Type",
    yaxis_title="Estimated Transaction Value (£)"
)

fig.update_yaxes(type="log")

fig.show()

## Average Transaction Value by Transaction Type

- **Purpose** — Compare the average monetary value of transactions across different transaction types.

- **AverageAmountGBP** — The mean GBP-equivalent transaction value for each transaction type. Enables:
  - Comparison of average transaction values across transaction types
  - Identification of transaction types with relatively high or low average values
  - Comparison of average transaction value with total transaction volume

- **Why it matters** — Helps distinguish transaction types that occur frequently from those that involve larger average values. This can support customer-behaviour analysis and inform further investigation into transaction patterns. Averages can be influenced by unusually large transactions, so they should be interpreted alongside transaction counts and value distributions.

In [ ]:
transaction_value = (
    cleaned_customer_transaction
    .groupby("TransactionType", as_index=False)
    .agg(
        AverageAmountGBP=("AmountGBP", "mean"),
        TransactionCount=("TransactionType", "size")
    )
    .sort_values("AverageAmountGBP", ascending=True)
)

fig = px.bar(
    transaction_value,
    x="AverageAmountGBP",
    y="TransactionType",
    orientation="h",
    text="AverageAmountGBP",
    title="Average Transaction Value by Transaction Type",
    color="AverageAmountGBP",
    color_continuous_scale="Viridis",
    hover_data=["TransactionCount"]
)

fig.update_traces(
    texttemplate="£%{text:,.2f}",
    textposition="outside"
)

fig.update_layout(
    height=550,
    xaxis_title="Average Transaction Value (£)",
    yaxis_title="Transaction Type",
    coloraxis_showscale=False
)

fig.show()